In [7]:
import pandas as pd

In [8]:
links = pd.read_csv("./article_links_12-09-25_02-14-55.csv")
links

,Unnamed: 0,type,time_ago,title,link
0,0,Fight Coverage,2 hours ago,Press Conference Highlights | Canelo vs Crawford,https://www.ufc.com/video/150879
1,1,Athletes,2 hours ago,Alexander Hernandez Feels Like He Has Nothing ...,https://www.ufc.com/news/alexander-hernandez-f...
2,2,Athletes,2 hours ago,Tatiana Suarez Is Ready To Show Her Best Self,https://www.ufc.com/news/tatiana-suarez-is-rea...
3,3,Fight Coverage,6 hours ago,Pre-Fight Press Conference | Canelo vs Crawford,https://www.ufc.com/video/150864
4,4,Licensed,7 hours ago,Discover The Latest Noche UFC Merch &amp; Gear...,https://www.ufc.com/news/discover-latest-noche...
...,...,...,...,...,...
4808,4808,Interviews,1 year ago,ChangHo Lee Octagon Interview | UFC Saudi Arabia,https://www.ufc.com/video/140129
4809,4809,Announcements,1 year ago,"Stancé Launches Globally, Teams Up with UFC to…",https://www.ufc.com/news/stance-launches-globa...
4810,4810,Athletes,1 year ago,A Dream Within Reach For ChangHo Lee,https://www.ufc.com/news/dream-within-reach-ch...
4811,4811,Fight Coverage,1 year ago,Cold Open | UFC Saudi Arabia,https://www.ufc.com/video/140128


In [9]:
athlete_links = links[links["type"] == "Athletes"]
athlete_links

,Unnamed: 0,type,time_ago,title,link
1,1,Athletes,2 hours ago,Alexander Hernandez Feels Like He Has Nothing ...,https://www.ufc.com/news/alexander-hernandez-f...
2,2,Athletes,2 hours ago,Tatiana Suarez Is Ready To Show Her Best Self,https://www.ufc.com/news/tatiana-suarez-is-rea...
10,10,Athletes,1 day ago,Dustin Stoltzfus: 'It's Now Or Never',https://www.ufc.com/news/dustin-stoltzfus-its-...
12,12,Athletes,1 day ago,Diego Lopes Knows He Is A High-Level Fighter,https://www.ufc.com/news/diego-lopes-knows-he-...
14,14,Athletes,1 day ago,"Zachary Reese | All Gas, No Brakes",https://www.ufc.com/news/zachary-reese-all-gas...
...,...,...,...,...,...
4781,4781,Athletes,1 year ago,People Are Rightfully Taking Notice Of Payton ...,https://www.ufc.com/news/people-taking-notice-...
4783,4783,Athletes,1 year ago,Jiří Procházka Career Highlights,https://www.ufc.com/news/jiri-prochazka-career...
4788,4788,Athletes,1 year ago,Joe Pyfer Continues To Grow,https://www.ufc.com/news/joe-pyfer-continues-t...
4810,4810,Athletes,1 year ago,A Dream Within Reach For ChangHo Lee,https://www.ufc.com/news/dream-within-reach-ch...


In [49]:
athlete_articles = athlete_links[athlete_links["link"].str.contains("news")]
athlete_articles

,Unnamed: 0,type,time_ago,title,link
1,1,Athletes,2 hours ago,Alexander Hernandez Feels Like He Has Nothing ...,https://www.ufc.com/news/alexander-hernandez-f...
2,2,Athletes,2 hours ago,Tatiana Suarez Is Ready To Show Her Best Self,https://www.ufc.com/news/tatiana-suarez-is-rea...
10,10,Athletes,1 day ago,Dustin Stoltzfus: 'It's Now Or Never',https://www.ufc.com/news/dustin-stoltzfus-its-...
12,12,Athletes,1 day ago,Diego Lopes Knows He Is A High-Level Fighter,https://www.ufc.com/news/diego-lopes-knows-he-...
14,14,Athletes,1 day ago,"Zachary Reese | All Gas, No Brakes",https://www.ufc.com/news/zachary-reese-all-gas...
...,...,...,...,...,...
4781,4781,Athletes,1 year ago,People Are Rightfully Taking Notice Of Payton ...,https://www.ufc.com/news/people-taking-notice-...
4783,4783,Athletes,1 year ago,Jiří Procházka Career Highlights,https://www.ufc.com/news/jiri-prochazka-career...
4788,4788,Athletes,1 year ago,Joe Pyfer Continues To Grow,https://www.ufc.com/news/joe-pyfer-continues-t...
4810,4810,Athletes,1 year ago,A Dream Within Reach For ChangHo Lee,https://www.ufc.com/news/dream-within-reach-ch...


In [11]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.remote.webelement import WebElement

In [18]:
import re

In [32]:
credit_pattern = (
    r"^By\s+(?P<name>[^,•]+)"
    r"(?:,\s+on\s+(?P<social_media>\w+):?\s+@(?P<handle>\S+))?"
    r"\s+•\s+(?P<date>.+)$"
)

In [ ]:
test_credit = "BY THOMAS GERBASI, ON X @TGERBASI • SEP. 12, 2025"

In [52]:
def scrape_article(link: str):
    driver = webdriver.Chrome()
    driver.get(link)

    try:
        title = driver\
            .find_element(By.CLASS_NAME, "field--name-node-title")\
            .find_element(By.TAG_NAME, "h1")\
            .get_attribute("innerHTML")\
            .strip()\
            .replace("\n", "")
        
        subtitle = driver\
            .find_element(By.CLASS_NAME, "field--name-subtitle")\
            .get_attribute("innerHTML")\
            .strip()\
            .replace("\n", "")
        
        credit = driver\
            .find_element(By.CLASS_NAME, "c-hero__article-credit")\
            .get_attribute("innerHTML")\
            .strip()\
            .replace("\n", "")

        name = None
        social_media = None
        handle = None
        date = None

        credit_match = re.search(credit_pattern, credit)
        if credit_match:
            name = credit_match.group("name")
            social_media = credit_match.group("social_media")
            handle = credit_match.group("handle")
            date = credit_match.group("date")
        
        content = driver.find_element(By.CLASS_NAME, "l-two-col__content")
        texts = [
            text.get_attribute("innerHTML").strip()
            for text in content.find_elements(By.CSS_SELECTOR, "p, a")
        ]

        return {
            'title': title,
            'subtitle': subtitle,
            'credit': credit,
            'name': name,
            'social_media': social_media,
            'handle': handle,
            'date': date,
            'texts': texts
        }

    finally:
        driver.quit()

In [54]:
test_article_scraped = scrape_article("https://www.ufc.com/news/alexander-hernandez-feels-like-he-has-nothing-to-lose-noche-ufc")
test_article_scraped

{'title': 'Alexander Hernandez Feels Like He Has Nothing To Lose',
 'subtitle': 'After A Massive Knockout Win, Alexander Hernandez Is Hoping To Give The San Antonio Crowd Plenty To Cheer About On September 13',
 'credit': 'By Thomas Gerbasi, on X @tgerbasi • Sep. 12, 2025',
 'name': 'Thomas Gerbasi',
 'social_media': 'X',
 'handle': 'tgerbasi',
 'date': 'Sep. 12, 2025',
 'texts': ['If Alexander Hernandez was like the rest of us, he might have scored a first-round knockout over Chase Hooper last month and sat out the rest of the year. Instead, the newly minted “El Gran Chango” will put his three-fight winning streak on the line against Diego Ferreira this week in San Antonio.',
  'Why?',
  "“Just strike while that mother**ker's hot,” he laughed.",
  '<a href="https://www.ufc.com/tunein"><strong>Tune-In Information For Noche UFC And Canelo vs Crawford</strong></a>',
  '<strong>Tune-In Information For Noche UFC And Canelo vs Crawford</strong>',
  'But when the possibility of taking that w

In [55]:
test_article_scraped["texts"]

['If Alexander Hernandez was like the rest of us, he might have scored a first-round knockout over Chase Hooper last month and sat out the rest of the year. Instead, the newly minted “El Gran Chango” will put his three-fight winning streak on the line against Diego Ferreira this week in San Antonio.',
 'Why?',
 "“Just strike while that mother**ker's hot,” he laughed.",
 '<a href="https://www.ufc.com/tunein"><strong>Tune-In Information For Noche UFC And Canelo vs Crawford</strong></a>',
 '<strong>Tune-In Information For Noche UFC And Canelo vs Crawford</strong>',
 'But when the possibility of taking that winning streak into 2026 while eating like a fat kid through Thanksgiving and Christmas was brought up, he admits it did cross his mind. But getting through the Hooper fight unscathed and feeling as good as he ever has in the Octagon made his decision to get right back to work the right one.',
 'Continue watching',
 'Upgrade licence',
 "“I went to bed, didn't think about it all the next